In [7]:
import torch
import numpy as np
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

Running on: cuda


PyTorch separates the logic into two distinct parts:

The Map (Dataset): "Here is my data, and here is how you get one specific item."

The Engine (DataLoader): "I will take that map, grab items in parallel, shuffle them, and stack them into batches."

### 1. The Dataset Class (The Map)
You create a class that inherits from torch.utils.data.Dataset. You must implement three magic methods:

__init__: The setup. (e.g., load a CSV of filenames, store a list of image paths). Do not load all heavy images into memory here unless the dataset is tiny.

__len__: Returns the total size of the dataset (int).

__getitem__: The workhorse. Given an index idx, it retrieves one sample and its label. This is where you read the image file, apply augmentation, or parse text.

In [2]:
from torch.utils.data import Dataset

class MyCustomDataset(Dataset):
    def __init__(self, num_samples, input_dim):
        # 1. Setup: Create dummy data in memory for this example
        # In real life, you would store paths like self.file_paths = glob.glob("*.jpg")
        self.x_data = torch.randn(num_samples, input_dim)
        self.y_data = torch.randint(0, 2, (num_samples,)) # Binary labels (0 or 1)

    def __len__(self):
        return len(self.x_data)

    def __getitem__(self, idx):
        # 3. Fetch ONE item
        # We can add logic here! Let's add noise dynamically.
        noise = torch.randn_like(self.x_data[idx]) * 0.1
        sample = self.x_data[idx] + noise
        label = self.y_data[idx]
        
        # Must return a tuple (features, label)
        return sample, label

### 2. The DataLoader (The Engine)
The Dataset just knows how to get item #5. The DataLoader orchestrates the training flow.

In [5]:
from torch.utils.data import DataLoader

# Instantiate the dataset
dataset = MyCustomDataset(num_samples=100, input_dim=10)

# Instantiate the loader
train_loader = DataLoader(
    dataset=dataset,
    batch_size=4,    # Stack 4 items into one tensor
    shuffle=True,    # Important for training! Reshuffles every epoch.
    num_workers=0    # Parallel sub-processes. Set to 2 or 4 on Linux/Windows for speed.
)

In [6]:
next(iter(train_loader))

[tensor([[ 1.2856, -0.7199,  1.1575, -1.4306,  0.6660,  1.6196,  1.4235, -0.8334,
          -0.6945, -0.7140],
         [ 0.1520, -0.3661, -0.4122,  0.4047, -0.8539,  2.6207, -1.0127,  0.3474,
          -0.6857, -0.5154],
         [-0.1070,  0.0425, -0.5639, -1.5575, -2.0274, -1.4265, -1.1188, -0.2878,
           1.1564,  2.2360],
         [-0.7362, -1.3617,  0.1745, -1.0416,  0.1903,  1.1601,  0.6938, -1.8177,
          -0.0404,  0.3467]]),
 tensor([0, 0, 1, 0])]

In [8]:
class SimpleNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(SimpleNet, self).__init__()
        # Layer 1: Input -> Hidden
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        # Layer 2: Hidden -> Output
        self.layer2 = nn.Linear(hidden_dim, output_dim)
        
        # Activation function (we can define it here or just use torch.relu in forward)
        self.relu = nn.ReLU()

    def forward(self, x):
        # Pass through Layer 1
        x = self.layer1(x)
        # Apply Activation
        x = self.relu(x)
        # Pass through Layer 2 (No activation on output for now)
        x = self.layer2(x)
        return x

class RandomDataset(Dataset):
    def __init__(self, num_samples, input_dim):
        self.len = num_samples
        # Create random features and random integer labels
        self.features = torch.randn(num_samples, input_dim)
        self.labels = torch.randint(0, 10, (num_samples,))

    def __len__(self):
        return self.len

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


# Configuration
INPUT_DIM = 20
HIDDEN_DIM = 64
OUTPUT_DIM = 10 # Example: 10 classes
BATCH_SIZE = 8

# 1. Instantiate Model
model = SimpleNet(INPUT_DIM, HIDDEN_DIM, OUTPUT_DIM)

# 2. Instantiate Data
dataset = RandomDataset(num_samples=100, input_dim=INPUT_DIM)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

print("--- Starting Execution ---")

# 3. Iterate ONE batch
# We use next(iter()) to grab the first batch immediately
features_batch, labels_batch = next(iter(loader))

print(f"Input Batch Shape: {features_batch.shape}") # Should be [8, 20]

# 4. Pass through Model
output = model(features_batch)

print(f"Output Batch Shape: {output.shape}") # Should be [8, 10]
print("\nSuccess! We have mapped (Batch, Input) -> (Batch, Output)")

--- Starting Execution ---
Input Batch Shape: torch.Size([8, 20])
Output Batch Shape: torch.Size([8, 10])

Success! We have mapped (Batch, Input) -> (Batch, Output)
